# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [43]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [44]:
df=pd.read_csv("data/AviationData.csv",encoding="latin1")
df.head()

/var/folders/bq/4m5y4fjn1t1_fpdlg7fvg6t80000gn/T/ipykernel_2454/2875895360.py:1: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("data/AviationData.csv",encoding="latin1")


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [45]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

In [46]:
df.isnull().sum().sort_values(ascending=False)

Schedule                  76307
Air.carrier               72241
FAR.Description           56866
Aircraft.Category         56602
Longitude                 54516
Latitude                  54507
Airport.Code              38757
Airport.Name              36185
Broad.phase.of.flight     27165
Publication.Date          13771
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Fatal.Injuries      11401
Engine.Type                7096
Report.Status              6384
Purpose.of.flight          6192
Number.of.Engines          6084
Total.Uninjured            5912
Weather.Condition          4492
Aircraft.damage            3194
Registration.Number        1382
Injury.Severity            1000
Country                     226
Amateur.Built               102
Model                        92
Make                         63
Location                     52
Investigation.Type            0
Event.Date                    0
Accident.Number               0
Event.Id                      0
dtype: i

In [47]:
df.columns

Index(['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date',
       'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code',
       'Airport.Name', 'Injury.Severity', 'Aircraft.damage',
       'Aircraft.Category', 'Registration.Number', 'Make', 'Model',
       'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description',
       'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries',
       'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured',
       'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status',
       'Publication.Date'],
      dtype='object')

In [48]:
df["Event.Date"]=pd.to_datetime(df["Event.Date"],errors="coerce")
df["Event.Date"].head()


0   1948-10-24
1   1962-07-19
2   1974-08-30
3   1977-06-19
4   1979-08-02
Name: Event.Date, dtype: datetime64[ns]

In [49]:
df["Total.Passengers"]=(df["Total.Fatal.Injuries"]+df["Total.Serious.Injuries"]+df["Total.Minor.Injuries"]+df["Total.Uninjured"])

In [50]:
df[["Total.Passengers","Total.Fatal.Injuries","Total.Serious.Injuries"]].head()

,Total.Passengers,Total.Fatal.Injuries,Total.Serious.Injuries
0,2.0,2.0,0.0
1,4.0,4.0,0.0
2,NaN,3.0,NaN
3,2.0,2.0,0.0
4,NaN,1.0,2.0


In [51]:
df["Total.Passengers"]=(df["Total.Fatal.Injuries"].fillna(0)+df["Total.Serious.Injuries"].fillna(0)+df["Total.Minor.Injuries"].fillna(0)+df["Total.Uninjured"].fillna(0))

In [52]:
df[["Total.Passengers","Total.Fatal.Injuries","Total.Serious.Injuries"]].head()

,Total.Passengers,Total.Fatal.Injuries,Total.Serious.Injuries
0,2.0,2.0,0.0
1,4.0,4.0,0.0
2,3.0,3.0,NaN
3,2.0,2.0,0.0
4,3.0,1.0,2.0


In [53]:
#fatality rate
df["Fatality.Rate"]=df["Total.Fatal.Injuries"]/df["Total.Passengers"]


In [54]:
#counter check
df[["Fatality.Rate","Total.Fatal.Injuries","Total.Passengers"]].head()

,Fatality.Rate,Total.Fatal.Injuries,Total.Passengers
0,1.000000,2.0,2.0
1,1.000000,4.0,4.0
2,1.000000,3.0,3.0
3,1.000000,2.0,2.0
4,0.333333,1.0,3.0


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [55]:
df=df[df["Event.Date"].dt.year>=1983]

In [56]:
df=df[df["Investigation.Type"]=="Accident"]

In [57]:
df.shape

(81534, 33)

Data cleaning and preparation
To prepare the data  i converveted the Event.Date column and only records from 1983 onwards were kept.missing injury values were replaced with 0 to enable calculations.Two new variables were created:Total.Passengers and Fatality.Rate which will help measure safety outcomes.

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [58]:
df["Total.Passengers"]=(df["Total.Fatal.Injuries"].fillna(0)+df["Total.Serious.Injuries"].fillna(0)+df["Total.Minor.Injuries"].fillna(0)+df["Total.Uninjured"].fillna(0))
df["Fatality.Rate"]=df["Total.Fatal.Injuries"]/df["Total.Passengers"].replace(0,1)

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [59]:
df["Aircraft.damage"]=df["Aircraft.damage"].str.strip().str.lower()
df["Destroyed"]=df["Aircraft.damage"].apply(lambda x: 1 if x=="destroyed" else 0)
df[["Aircraft.damage","Destroyed"]].head()

,Aircraft.damage,Destroyed
3600,NaN,0
3601,substantial,0
3602,substantial,0
3603,substantial,0
3604,substantial,0


The Aircraft.damage colum was standardized by converting values to lower case and removing spaces.i introduced a new variable destroyed  to indicate whether the aircraft was completly destroyed or not

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [60]:
df["Make"].value_counts().head(20)

Make
Cessna         20660
Piper          11179
CESSNA          4820
Beech           3922
PIPER           2799
Bell            1984
Mooney          1019
BEECH           1007
Grumman          977
Robinson         920
Boeing           882
Bellanca         821
Hughes           737
Schweizer        610
Air Tractor      575
BELL             569
Aeronca          457
BOEING           446
Maule            426
Champion         415
Name: count, dtype: int64

In [61]:
df["Make"]=df["Make"].str.strip().str.lower()
df["Make"].value_counts().head(20)

Make
cessna            25480
piper             13978
beech              4929
bell               2553
boeing             1328
mooney             1256
robinson           1196
grumman            1055
bellanca            978
hughes              872
schweizer           753
air tractor         671
aeronca             606
maule               570
champion            505
stinson             418
aero commander      405
luscombe            391
taylorcraft         366
de havilland        364
Name: count, dtype: int64

In [62]:
common_makes=df["Make"].value_counts()
common_makes=common_makes[common_makes>=50].index
df["Make_clean"]=df["Make"].apply(lambda x: x if x in common_makes else "other")
df["Make_clean"].value_counts().head()

Make_clean
cessna    25480
piper     13978
other     13860
beech      4929
bell       2553
Name: count, dtype: int64

I cleaned the make column by standandardizing text to lower case and removing inconsistencies.

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [63]:
df["Model"]=df["Model"].fillna("unknown")

In [64]:
df["Model"]=df["Model"].str.strip().str.lower()

In [65]:
df["Model"].value_counts().head(10)

Model
152          2218
172          1648
172n         1095
pa-28-140     868
172m          755
150           745
172p          661
182           612
180           595
pa-18-150     557
Name: count, dtype: int64

In [66]:
df["Aircraft_Type"]=df["Make_clean"]+"_"+df["Model"]

In [67]:
df[["Make_clean","Model","Aircraft_Type"]].head()

,Make_clean,Model,Aircraft_Type
3600,other,ax-6,other_ax-6
3601,cessna,182p,cessna_182p
3602,cessna,182rg,cessna_182rg
3603,cessna,182p,cessna_182p
3604,piper,pa-28r-200,piper_pa-28r-200


The model columnn was cleaned by filling missing values and standardizing text.since model names are not uniques across manufacturers a new variable combining make and model was created to make unique aircraft types.

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [68]:
cols=["Engine.Type","Weather.Condition","Broad.phase.of.flight","Purpose.of.flight"]
for col in cols:
    df[col]=df[col].str.strip().str.lower()

The selected categorical columns  were standardized by converting text to lowercase and removing extra spaces to ensure consistencies


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [69]:
missing=df.isnull().mean().sort_values(ascending=False)
missing.head(10)

Schedule                 0.881927
Air.carrier              0.822234
FAR.Description          0.669806
Aircraft.Category        0.667758
Longitude                0.590527
Latitude                 0.590416
Airport.Code             0.428913
Airport.Name             0.404236
Broad.phase.of.flight    0.309159
Publication.Date         0.160264
dtype: float64

In [70]:
df=df.drop(columns=["Schedule","Air.carrier","FAR.Description"],errors="ignore")

Columns with a high proportion of missing values were removed to improve data quality and reduce noise in the analysis.

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [71]:
df.to_csv("cleaned_aviation_data.csv",index=False)